# Libaries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd

from src.benchmark import (
    ElasticityConfig,
    ModelingDatasetBuilder,
    BenchmarkFairFormulaBuilder,
    LogLogBenchmarkFairModel,
    ElasticityBenchmarkFairPipeline,
)

from src.dominick import DominickDataLoader

In [3]:
TRAIN_FRAC = 0.8

# Loader

In [4]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()

print(f"Dataset shape: {df.shape}")
print(f"N semanas: {df['week_id'].nunique()}")

Dataset shape: (463722, 30)
N semanas: 302


In [5]:
sorted_weeks   = sorted(df["week_id"].unique())
week_threshold = sorted_weeks[int(len(sorted_weeks) * TRAIN_FRAC)]

benchmark_train_df = df[df["week_id"] < week_threshold].copy()
benchmark_val_df   = df[df["week_id"] >= week_threshold].copy()

print("week_threshold:", week_threshold)
print("Train shape:", benchmark_train_df.shape)
print("Val shape:", benchmark_val_df.shape)

print(
    "Train week range:",
    benchmark_train_df["week_id"].min(),
    benchmark_train_df["week_id"].max(),
)
print(
    "Val week range:",
    benchmark_val_df["week_id"].min(),
    benchmark_val_df["week_id"].max(),
)

week_threshold: 339
Train shape: (439146, 30)
Val shape: (24576, 30)
Train week range: 91 338
Val week range: 339 399


# Config

In [6]:
config = ElasticityConfig(csv_path="elasticity_dataset.csv")

# Pipeline

In [7]:
pipeline = ElasticityBenchmarkFairPipeline(
    config=config,
    dataset_builder=ModelingDatasetBuilder(config),
    formula_builder=BenchmarkFairFormulaBuilder(config),
)

benchmark_results = pipeline.run(
    train_df=benchmark_train_df,
    val_df=benchmark_val_df,
)

In [8]:
display(benchmark_results.head())
display(benchmark_results["status"].value_counts(dropna=False))

,store_code,upc_code,status,n_train,n_val,elasticity,elasticity_se,elasticity_ci_low,elasticity_ci_high,elasticity_p_value,mae_val,rmse_val,r2_val
0,5,5230000035,ok,133,11,-0.007815,2.460816,-4.830926,4.815296,9.974660e-01,0.554984,0.654422,-0.127168
1,5,7289000011,ok,124,14,1.240572,2.914477,-4.471698,6.952841,6.703568e-01,0.424666,0.531537,0.400615
2,8,1820000784,ok,235,61,-4.303493,0.891046,-6.049912,-2.557074,1.367341e-06,0.487062,0.656843,0.106730
3,8,3410010505,ok,234,61,-3.100294,0.547520,-4.173413,-2.027175,1.492421e-08,0.570903,0.695974,-0.540952
4,8,5230000035,ok,170,59,-1.632433,1.314376,-4.208562,0.943697,2.142428e-01,0.513768,0.683454,0.170350


status
ok                          440
no_price_variation_train     58
insufficient_train_obs       21
Name: count, dtype: int64

In [9]:
benchmark_ok = benchmark_results[benchmark_results["status"] == "ok"].copy()

print("Series válidas:", len(benchmark_ok))
display(benchmark_ok[["mae_val", "rmse_val", "r2_val"]].describe())
display(benchmark_ok["elasticity"].describe())

Series válidas: 440


,mae_val,rmse_val,r2_val
count,440.000000,440.000000,426.000000
mean,0.771698,0.914952,-6.009610
std,1.724409,1.818982,43.619997
min,0.013216,0.013216,-560.126448
25%,0.452890,0.573430,-0.371490
50%,0.525103,0.656896,0.005233
75%,0.622555,0.774251,0.224534
max,23.142286,24.657975,0.839929


count    440.000000
mean      -3.113071
std        2.789086
min      -16.421731
25%       -4.618545
50%       -3.273671
75%       -1.824958
max       16.998416
Name: elasticity, dtype: float64

# Save

In [10]:
benchmark_ok.to_csv("../data/benchmark_elasticities_store_upc.csv", index=False)
benchmark_ok.describe()

,store_code,upc_code,n_train,n_val,elasticity,elasticity_se,elasticity_ci_low,elasticity_ci_high,elasticity_p_value,mae_val,rmse_val,r2_val
count,440.000000,4.400000e+02,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,4.400000e+02,440.000000,440.000000,426.000000
mean,89.486364,6.295032e+09,188.070455,50.859091,-3.113071,1.441208,-5.937788,-0.288355,1.214653e-01,0.771698,0.914952,-6.009610
std,36.059996,2.311195e+09,54.742154,17.234102,2.789086,1.477191,3.525590,4.460164,2.362181e-01,1.724409,1.818982,43.619997
min,5.000000,1.820001e+09,31.000000,1.000000,-16.421731,0.333270,-30.418532,-7.107024,1.887612e-63,0.013216,0.013216,-560.126448
25%,72.750000,3.410011e+09,174.000000,50.000000,-4.618545,0.613069,-6.984302,-2.700170,6.972493e-09,0.452890,0.573430,-0.371490
50%,96.500000,7.289000e+09,203.000000,59.000000,-3.273671,0.932297,-5.405613,-1.013533,1.661997e-03,0.525103,0.656896,0.005233
75%,116.000000,8.248812e+09,231.000000,61.000000,-1.824958,1.605089,-3.891871,0.859731,1.014623e-01,0.622555,0.774251,0.224534
max,139.000000,9.893110e+09,241.000000,61.000000,16.998416,15.011697,1.245465,40.984722,9.974660e-01,23.142286,24.657975,0.839929
